In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1" # for debugging

import torch
import torch.nn as nn
import math

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
class SimpleGPT(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()
        self.d_model = d_model

        # Single attention head
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.attn_out = nn.Linear(d_model, d_model)

        # Feed-forward
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

        # Layer norms
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Output
        self.output_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        _, seq_len, _ = x.shape

        # Self-attention with causal mask
        residual = x
        x = self.ln1(x)

        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_model)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        attn = attn.masked_fill(causal_mask, float('-inf'))
        attn = torch.softmax(attn, dim=-1)

        x = attn @ v
        x = self.attn_out(x)
        x = residual + x

        # Feed-forward
        x = x + self.ffn(self.ln2(x))

        # Output projection
        return self.output_proj(x)


class SimpleDequant(nn.Module):
    def __init__(self, max_seq_len: int, num_instruments: int):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.num_instruments = num_instruments

        self.gpt = SimpleGPT(
            d_model=num_instruments * 3,
            max_seq_len=max_seq_len,
        )
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, num_instruments, hov_size = x.shape

        assert seq_len <= self.max_seq_len, "max_seq_len exceeded."
        assert num_instruments == self.num_instruments
        assert hov_size == 3

        # Flatten each timestep to a single dimension
        x = x.flatten(start_dim=2)

        # Run the GPT
        y: torch.Tensor = self.gpt(x)

        # Reshape the output tensor
        y = y.reshape(batch_size, seq_len, num_instruments, hov_size)

        # Apply different activations to hits, offsets, and velocities
        hits = self.sigmoid(y[..., 0:1])
        offsets = 0.5 * self.tanh(y[..., 1:2])
        velocities = self.sigmoid(y[..., 2:3])

        # Concatenate along last dimension
        return torch.cat([hits, offsets, velocities], dim=-1)

In [ ]:
# basic test pattern
pattern = torch.tensor([
    [[1.0, 0.1, 0.9], [0.0, 0.0, 0.0]],
    [[0.0, 0.0, 0.0], [1.0, 0.1, 0.5]],
    [[1.0, -0.1, 0.9], [1.0, -0.1, 0.9]],
    [[0.0, 0.0, 0.0], [1.0, 0.1, 0.5]],
])

# create multiple different samples by shifting the test pattern
xs = torch.zeros((4, 3, 2, 3))
ys = torch.zeros((4, 3, 2, 3))
for i in range(4):
    shifted_pattern = torch.roll(pattern, i, dims=0)
    xs[i] = shifted_pattern[:-1]
    ys[i] = shifted_pattern[1:]

# create a basic optimizer
model = SimpleDequant(4, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

print("Training...")
for epoch in range(1000):
    # use the whole dataset as a single batch
    output = model(xs)
    loss = criterion(output, ys)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

torch.set_printoptions(precision=2, sci_mode=False)
print("\nTesting prediction:")
print("Input (first 3 steps):")
print(xs[0])
print("\nTarget (last 3 steps):")
print(ys[0])
print("\nPrediction:")
print(model(xs[0].unsqueeze(0))[0])